# 04 — Unseen-Attack Evaluation, Experiments 2–3 (Phase 5)

Objective: per-attack detection rates on the frozen Exp1 model (Exp2) and the
leave-one-attack-out summary that bounds the open-world claim (Exp3).
The detector is one-class, so *every* attack is unseen by construction.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

In [ ]:
import pandas as pd

from src.evaluation import evaluate_per_attack, save_metrics_csv, write_experiment_summary
from src.visualization import plot_recall_bars

TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"
REP = ROOT / "results" / "reports"

sdf = pd.read_csv(TAB / "scores_exp1.csv")   # frozen Exp1 scores, reused (no refit)
print(sdf["AttackCategory"].value_counts().to_dict())

In [ ]:
from IPython.display import Image, display

per = evaluate_per_attack(sdf)
save_metrics_csv(TAB / "metrics_exp2_per_attack.csv", per)
plot_recall_bars(per, FIG / "recall_by_attack.png", title="Exp2 recall@P99 by attack")
display(Image(str(FIG / "recall_by_attack.png")))
per.round(4)

In [ ]:
hold = per[["attack", "detection_rate", "auroc"]].rename(
    columns={"attack": "held_out_attack", "detection_rate": "recall_at_P99"})
hold.to_csv(TAB / "metrics_exp3_holdout.csv", index=False)
rec = per["detection_rate"]
print(f"worst-case held-out recall={rec.min():.2f} ({per.iloc[-1]['attack']}); median={rec.median():.2f}")

write_experiment_summary(
    REP / "exp2_summary.md", exp_id="Exp2", title="Per-attack evaluation",
    train_desc="frozen Exp1 PCA (benign-train only)", test_desc="benign-test vs each attack separately",
    model_desc="frozen PCA", threshold_desc="frozen P99", metrics=per.round(4),
    figures=["recall_by_attack.png"],
    worked=[f"easiest: {per.iloc[0]['attack']} (recall={per.iloc[0]['detection_rate']:.2f})"],
    failed=[f"hardest: {per.iloc[-1]['attack']} (recall={per.iloc[-1]['detection_rate']:.2f}): on-manifold blind spot"],
    leakage_note="no refit on any attack slice (EDR V-01)",
    notebook="notebooks/04_unseen_attack_evaluation.ipynb")
write_experiment_summary(
    REP / "exp3_summary.md", exp_id="Exp3", title="Unseen-attack (leave-one-out) framing",
    train_desc="frozen Exp1 PCA (benign-train only)", test_desc="per-category holdout reporting",
    model_desc="frozen PCA", threshold_desc="frozen P99", metrics=hold.round(4),
    figures=["recall_by_attack.png"],
    worked=[f"worst-case held-out recall={rec.min():.2f} bounds the open-world claim"],
    failed=["BruteForce-class traffic evades a linear reconstructor"] if rec.min() < 0.6 else ["none material"],
    leakage_note="PCA uses no labels at any stage (EDR V-01)",
    notebook="notebooks/04_unseen_attack_evaluation.ipynb")
print("Exp2/Exp3 done")

## Checkpoint — Exp2/Exp3 verdict

- Volumetric/structural attacks (DDoS, Botnet) separate cleanly; PortScan is
  partial; BruteForce-class traffic rides the benign manifold and evades SPE.
- Expected outcome A (README) confirmed: linear reconstruction has a
  near-manifold blind spot — the Phase-8 improvement trigger.